#Importing All Dependencies

In [100]:
##Installing PyTorch Geometric
!pip install torch_geometric
!pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.2.0+cu121.html



Looking in links: https://data.pyg.org/whl/torch-2.2.0+cu121.html


In [101]:
!pip install implicit

In [102]:
## DOUBLE CHECKING THAT IT WORKED
import torch_geometric
print(torch_geometric.__version__)


2.7.0


In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data
from torch_geometric.utils import k_hop_subgraph, to_undirected
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from model_specifications import *
from subgraph_dataclass import *
from retail_data_prep import preprocess_events


# Prepare and Clean Dataset 

* Download from kaggle
* Filter out users and items with very few interactions to reduce noise
* Convert/clean timestamps
* Remap the item/user ids to node indicies

In [ ]:
events = preprocess_events(limit = None)


In [109]:
## for MFmapping
mf_user_ids = events['visitorid'].astype('category').cat.codes
mf_item_ids = events['itemid'].astype('category').cat.codes

events['mf_user'] = mf_user_ids
events['mf_item'] = mf_item_ids

num_mf_users = events['mf_user'].max() + 1
num_mf_items = events['mf_item'].max() + 1


### Mapping Users and Items to Node Indices

In [ ]:

USE_EDGE_WEIGHTS = True

event_weight_map = {
    'view': 1.0,
    'addtocart': 2.0,
    'transaction': 3.0
}

unique_users = events['visitorid'].unique()
unique_items = events['itemid'].unique()
num_nodes = unique_items + unique_users

if USE_EDGE_WEIGHTS:
    events['weight'] = events['event'].map(event_weight_map).fillna(1.0)
else:
    events['weight'] = 1.0

edge_src = torch.tensor(events['user_idx'].values, dtype=torch.long)
edge_dst = torch.tensor(events['item_idx'].values, dtype=torch.long)
edge_index = torch.stack([edge_src, edge_dst], dim=0)
edge_weight = torch.tensor(events['weight'].values, dtype=torch.float)

deg = torch.zeros(num_nodes, dtype=torch.float)
#deg.index_add_(0, edge_src, torch.ones_like(edge_src, dtype=torch.float))
#deg.index_add_(0, edge_dst, torch.ones_like(edge_dst, dtype=torch.float))

graph_summary_data = Data(
    edge_index=edge_index,
    edge_weight=edge_weight,
    x=deg.view(-1, 1),
    num_nodes=num_nodes
)





Users: 16675 | Items: 2457 | Total nodes: 19132
Data(x=[19132, 1], edge_index=[2, 43903], edge_weight=[43903], num_nodes=19132)


# Test/Train split

In [ ]:
def get_preferred_negatives(events_subset):

    transaction_events = events_subset[events_subset['event'] == 'transaction']
    transactional_visitors = transaction_events['user_idx'].unique()
    view_events_df = events_subset[events_subset['event'] == 'view'][['user_idx', 'item_idx']].drop_duplicates()
    view_events_by_transactional_visitors = view_events_df[view_events_df['user_idx'].isin(transactional_visitors)]
    addtocart_events_df = events_subset[events_subset['event'] == 'addtocart'][['user_idx', 'item_idx']].drop_duplicates()
    merged_df = view_events_by_transactional_visitors.merge(
        addtocart_events_df.assign(is_addtocart=True),
        on=['user_idx', 'item_idx'],
        how='left',
        indicator=True
    )

    view_but_no_addtocart_events = merged_df[merged_df['_merge'] == 'left_only'][['user_idx', 'item_idx']]
    return view_but_no_addtocart_events

def random_negative_edges(num_samples, positive_sample, events_subset):
    
    user_set = events_subset['user_idx'].unique()
    item_set = events_subset['item_idx'].unique()

    srcs = []
    dsts = []
    
    while len(srcs) < num_samples:
        u = np.random.choice(user_set)
        i = np.random.choice(item_set)
        if positive_sample.query("user_idx == @u and item_idx == @i").empty:
            srcs.append(u)
            dsts.append(i)
    return pd.DataFrame({'user_idx': srcs, 'item_idx': dsts})


def generate_negative_sample(positive_sample, events_subset, neg_to_pos_ratio = 1):

    target_count = np.floor(positive_sample.shape[0] * neg_to_pos_ratio)

    min_time = positive_sample['timestamp'].min()
    max_time = positive_sample['timestamp'].max()
    fake_timestamps = np.random.random_integers(min_time, max_time, target_count)

    negative_starter = get_preferred_negatives(events_subset)


    if target_count <= negative_starter.shape[0]:
        
        negative_sample = negative_starter.sample(target_count)
        negative_sample['timestamp'] = fake_timestamps
        return negative_sample
    
    num_additional_negatives = target_count - negative_starter.shape[0]

    additional_negatives = random_negative_edges(num_additional_negatives, positive_sample, events_subset)

    negative_sample = pd.concat([negative_starter, additional_negatives], axis=0)
    negative_sample['timestamp'] = fake_timestamps
    return negative_sample





In [ ]:


train_months = [5,6,7]
test_months = [8]
val_months = [9]

# get the events dataframes for the seperate time periods

train_events = events[events['month'].isin(train_months)]
test_events = events[events['month'].isin(test_months)]
val_events = events[events['month'].isin(val_months)]


# sample addtocarts/transactions for test set
positive_sample = events.query("event == 'addtocart' or event == 'transaction'")

pos_train_df = positive_sample[positive_sample['month'].isin(train_months)]
pos_test_df = positive_sample[positive_sample['month'].isin(test_months)]
pos_val_df = positive_sample[positive_sample['month'].isin(val_months)]

train_edges = pos_train_df[['user_idx', 'item_idx', 'timestamp']]
val_edges   = pos_val_df[['user_idx', 'item_idx', 'timestamp']]
test_edges  = pos_test_df[['user_idx', 'item_idx', 'timestamp']]

subsample = False

if subsample:
  print("Subsampled edges for GNN:")

  train_edges = train_edges[:5000]
  val_edges   = val_edges[:2000]
  test_edges  = test_edges[:2000]

  print("  Train:", len(train_edges))
  print("  Val  :", len(val_edges))
  print("  Test :", len(test_edges))

# get the negative edges
train_neg_edges = get_preferred_negatives(pos_train_df)
val_neg_edges   = get_preferred_negatives(pos_val_df)
test_neg_edges  = get_preferred_negatives(pos_test_df)


Temporal 60/20/20 Split:
  Train: 31868
  Val  : 3964
  Test : 8071


# Negative Sampling (Tentative)

Probably not a large enough space to negatively sample from just here.

#DRNL helper from Zhang and Chen

# Collating for DataLoader

In [ ]:

def collate_subgraphs(batch):
    xs, eis, ys = zip(*batch)

    ys = torch.stack(ys, dim=0)

    new_x = []
    new_edge_index = []
    # track which subgraph the data is from
    new_batch = []

    node_offset = 0
    subgraph_offset = 0

    for x_sub, ei_sub, in zip(xs, eis):
        n = x_sub.size(0)
        new_x.append(x_sub)
        new_edge_index.append(ei_sub + node_offset)
        b_sub = torch.zeros(x_sub.size(0), dtype=torch.long)
        new_batch.append(b_sub + subgraph_offset)
        
        node_offset += n
        subgraph_offset = new_batch[-1].max().item() + 1

    new_x = torch.cat(new_x, dim=0)
    new_edge_index = torch.cat(new_edge_index, dim=1)
    new_batch = torch.cat(new_batch, dim=0)

    return new_x, new_edge_index, new_batch, ys

# Building datasets

In [ ]:
train_dataset = LinkSubgraphDataset(train_edges, train_neg_edges, graph_summary_data, h=3)
val_dataset   = LinkSubgraphDataset(val_edges,   val_neg_edges,   graph_summary_data, h=3)
test_dataset  = LinkSubgraphDataset(test_edges,  test_neg_edges,  graph_summary_data, h=3)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,
                          collate_fn=collate_subgraphs)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False,
                          collate_fn=collate_subgraphs)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False,
                          collate_fn=collate_subgraphs)

len(train_dataset), len(val_dataset), len(test_dataset)


(63736, 7928, 16142)

GCN Baseline + Hybrid

# Creating model

In [127]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    all_logits = []
    all_labels = []
    total_loss = 0.0

    for x, edge_index, batch, y in loader:
        x = x.to(device)
        edge_index = edge_index.to(device)
        batch = batch.to(device)
        y = y.to(device)

        if is_train:
            optimizer.zero_grad()

        logits = model(x, edge_index, batch)
        loss = criterion(logits, y)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * y.size(0)
        all_logits.append(logits.detach().cpu())
        all_labels.append(y.detach().cpu())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    probs = torch.sigmoid(all_logits).numpy()
    preds = (probs >= 0.5).astype(int)
    true = all_labels.numpy().astype(int)

    try:
        auc = roc_auc_score(true, probs)
    except ValueError:
        auc = np.nan

    f1 = f1_score(true, preds)
    acc = accuracy_score(true, preds)
    avg_loss = total_loss / len(true)

    return avg_loss, auc, f1, acc


#Sample Batch (Having Issues)

In [128]:
in_channels = 2  ##for the dnrl labels
num_epochs = 5

gcn_model = BaselineGCNSubgraphEncoder(in_channels=in_channels).to(device)
gcn_opt = torch.optim.Adam(gcn_model.parameters(), lr=1e-3)

gat_model = GATOnlySubgraphEncoder(in_channels=in_channels).to(device)
gat_opt = torch.optim.Adam(gat_model.parameters(), lr=1e-3)

hybrid_model = PGADRLSubgraphEncoder(in_channels=in_channels).to(device)
hybrid_opt = torch.optim.Adam(hybrid_model.parameters(), lr=1e-3)


print("GCN Training")
for epoch in range(1, num_epochs + 1):
    train_loss, train_auc, train_f1, train_acc = run_epoch(train_loader, gcn_model, gcn_opt)
    val_loss, val_auc, val_f1, val_acc = run_epoch(val_loader, gcn_model, optimizer=None)
    print(f"[GCN] Epoch {epoch:02d} | Train loss={train_loss:.4f}, AUC={train_auc:.4f}, F1={train_f1:.4f}, ACC={train_acc:.4f}")
    print(f"Val   loss={val_loss:.4f}, AUC={val_auc:.4f}, F1={val_f1:.4f}, ACC={val_acc:.4f}")

print("GAT Training")
for epoch in range(1, num_epochs + 1):
    train_loss, train_auc, train_f1, train_acc = run_epoch(train_loader, gat_model, gat_opt)
    val_loss, val_auc, val_f1, val_acc = run_epoch(val_loader, gat_model, optimizer=None)
    print(f"[GAT] Epoch {epoch:02d} | Train loss={train_loss:.4f}, AUC={train_auc:.4f}, F1={train_f1:.4f}, ACC={train_acc:.4f}")
    print(f"Val   loss={val_loss:.4f}, AUC={val_auc:.4f}, F1={val_f1:.4f}, ACC={val_acc:.4f}")

print("Hybrid Training")
for epoch in range(1, num_epochs + 1):
    train_loss, train_auc, train_f1, train_acc = run_epoch(train_loader, hybrid_model, hybrid_opt)
    val_loss, val_auc, val_f1, val_acc = run_epoch(val_loader, hybrid_model, optimizer=None)
    print(f"[Hybrid] Epoch {epoch:02d} | Train loss={train_loss:.4f}, AUC={train_auc:.4f}, F1={train_f1:.4f}, ACC={train_acc:.4f}")
    print(f"Val   loss={val_loss:.4f}, AUC={val_auc:.4f}, F1={val_f1:.4f}, ACC={val_acc:.4f}")



GCN Training
[GCN] Epoch 01 | Train loss=53506.4039, AUC=0.9907, F1=0.9805, ACC=0.9802
Val   loss=0.0165, AUC=0.9995, F1=0.9946, ACC=0.9946
[GCN] Epoch 02 | Train loss=0.0118, AUC=0.9995, F1=0.9973, ACC=0.9973
Val   loss=0.0069, AUC=0.9997, F1=0.9986, ACC=0.9986


KeyboardInterrupt: 

Test Model

In [ ]:
gcn_test_loss, gcn_test_auc, gcn_test_f1, gcn_test_acc = run_epoch(
    test_loader, gcn_model, optimizer=None)
print("GCN:")
print(f"Loss: {gcn_test_loss:.4f}")
print(f"AUC : {gcn_test_auc:.4f}")
print(f"F1  : {gcn_test_f1:.4f}")
print(f"ACC : {gcn_test_acc:.4f}")


gat_test_loss, gat_test_auc, gat_test_f1, gat_test_acc = run_epoch(
    test_loader, gat_model, optimizer=None)
print("\nGAT:")
print(f"Loss: {gat_test_loss:.4f}")
print(f"AUC : {gat_test_auc:.4f}")
print(f"F1  : {gat_test_f1:.4f}")
print(f"ACC : {gat_test_acc:.4f}")


hybrid_test_loss, hybrid_test_auc, hybrid_test_f1, hybrid_test_acc = run_epoch(
    test_loader, hybrid_model, optimizer=None)
print("\nHybrid:")
print(f"Loss: {hybrid_test_loss:.4f}")
print(f"AUC : {hybrid_test_auc:.4f}")
print(f"F1  : {hybrid_test_f1:.4f}")
print(f"ACC : {hybrid_test_acc:.4f}")
